In [3]:
!pip install -q yt-dlp openai-whisper ffmpeg-python



In [4]:
# 2) Helper: try to download subtitles (manual or auto) with yt-dlp and parse .vtt
import os, glob, re, subprocess, json, shutil

def download_subs_ytdlp(url, lang='en'):
    """
    Download subtitles (manual or auto) using yt-dlp.
    Returns path to a .vtt/.srt file or None.
    """
    # request both manual and auto subs, prefer manual if available
    cmd = f'yt-dlp --skip-download --write-sub --write-auto-sub --sub-lang {lang} --sub-format vtt/srv3 --output "%(id)s.%(ext)s" "{url}"'
    print("Running:", cmd)
    subprocess.run(cmd, shell=True, check=False, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    # find any vtt/srt in cwd
    files = [f for f in glob.glob("*.vtt") + glob.glob("*.srt")]
    if not files:
        return None
    # pick newest file
    files.sort(key=os.path.getmtime, reverse=True)
    return files[0]

def vtt_to_text(vtt_path):
    """Simple VTT (or SRT) -> plain text extractor"""
    with open(vtt_path, 'r', encoding='utf-8', errors='ignore') as fh:
        lines = fh.readlines()
    text_lines = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if re.match(r'^\d+$', line):  # srt index
            continue
        if '-->' in line:  # timestamp line
            continue
        # drop VTT headers like "WEBVTT"
        if line.upper().startswith("WEBVTT"):
            continue
        text_lines.append(line)
    return " ".join(text_lines)

# Example:
url = "https://youtu.be/sQ56ve39l2I"  # replace with your URL
sub_file = download_subs_ytdlp(url, lang='en')
if sub_file:
    print("Found subtitle file:", sub_file)
    txt = vtt_to_text(sub_file)
    print("Transcript (preview):\n", txt[:500])
else:
    print("No subtitles found via yt-dlp.")


Running: yt-dlp --skip-download --write-sub --write-auto-sub --sub-lang en --sub-format vtt/srv3 --output "%(id)s.%(ext)s" "https://youtu.be/sQ56ve39l2I"
Found subtitle file: sQ56ve39l2I.en.vtt
Transcript (preview):
 Kind: captions Language: en Today we have the thinnest smartphone Apple has ever made, the new iPhone Air. It's fair as the Air enters my lair that Tim Cook sayin a prayer. It's got to be a scare to arrive at my secretar where I've got no hair, though I do declare that I will be fair. Feel free to stare while we disregard welfare. I am aware that this is a phone's worst nightmare. While the Air is kind of rare, I do have one spare. It's received much fanfare for its dramatic flare. It's time to 
